In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    "dbname": "sansilvestre_db",
    "user": "sansilvestre_user",
    "password":"secret",
    "host": "localhost",
    "port": 5433, #En el dockercompose es 5433 en vez de 5432 para no entrar en conflicto con un servicio postgres en local.
}

CSV_PATH = "./san_silvestre_silver.csv"  # ajustar ruta

In [2]:
df = pd.read_csv(CSV_PATH, sep=";")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40838 entries, 0 to 40837
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   edicion           40838 non-null  str    
 1   anho              40838 non-null  int64  
 2   carrera           40838 non-null  str    
 3   puesto            40838 non-null  int64  
 4   nombre            40838 non-null  str    
 5   dorsal            40838 non-null  int64  
 6   apellidos         40838 non-null  str    
 7   puesto_sexo       40761 non-null  float64
 8   puesto_categoria  40397 non-null  float64
 9   tiempo            40838 non-null  str    
 10  distancia_m       25899 non-null  float64
 11  ritmo_oficial     25899 non-null  str    
 12  ritmo_neto        23289 non-null  str    
 13  tiempo_neto       23289 non-null  str    
 14  categoria         40398 non-null  str    
 15  sexo              40776 non-null  str    
dtypes: float64(3), int64(3), str(10)
memory usage: 5.0 

In [4]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

In [5]:
df_ediciones = (
    df[["edicion", "anho"]]
    .drop_duplicates()
    .rename(columns={"edicion": "nombre", "anho": "ano"})
    .reset_index(drop=True)
)

In [6]:
for _, row in df_ediciones.iterrows():
    cur.execute(
        """
        INSERT INTO edicion (nombre, ano)
        VALUES (%s, %s)
        """,
        (row["nombre"], int(row["ano"])),
    )
conn.commit()